# No-U-Turn Sampler (NUTS)

NUTS removes the need to hand-tune the trajectory length in Hamiltonian Monte Carlo. At each step it doubles a balanced binary tree of leapfrog states forward or backward in time, stopping as soon as the trajectory "U-turns" on itself, then samples one state from the explored set, weighted by $\exp(-H)$.

We use the multinomial-sampling variant (Betancourt 2016) and adapt the step size via dual averaging (Nesterov 2009; Hoffman & Gelman 2014) during a warmup phase.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import torch
from torch.distributions.multivariate_normal import MultivariateNormal
import matplotlib.pyplot as plt

from mcmc_samplers import HamiltonianMonteCarlo, NUTS, SampleVisualizer, diagnostics

In [ ]:
class Banana:
    def __init__(self):
        self.mvn = MultivariateNormal(torch.zeros(2), torch.tensor([[1.0, 0.9], [0.9, 1.0]]))
    def __call__(self, x):
        y = torch.stack([x[..., 0], x[..., 1] + (x[..., 0] + 1.0) ** 2], dim=-1)
        return self.mvn.log_prob(y)

target = Banana()
x0 = torch.tensor([0.0, -1.0])

## Run NUTS with dual-averaging warmup

The first `warmup` iterations adapt the step size to hit the target acceptance probability of 0.8. After warmup the step size is locked at the running smoothed average.

In [ ]:
warmup = 500
N = 3500

nuts = NUTS(target=target, x0=x0, step_size=0.5, warmup=warmup, target_accept=0.8, max_depth=8, generator=torch.Generator().manual_seed(0))
step_size_trace = []
tree_depth_trace = []

def callback(step, sample, accepted):
    step_size_trace.append(nuts.dynamics.step_size)
    tree_depth_trace.append(nuts.last_tree_depth)

samples, _ = nuts(N, burn_in=warmup, callback=callback)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
axs[0].plot(step_size_trace)
axs[0].axvline(warmup, linestyle='--', color='k', alpha=0.4)
axs[0].set_title('NUTS step size (dual averaging)')
axs[0].set_xlabel('Iteration')
axs[1].plot(tree_depth_trace)
axs[1].set_title('Tree depth per step')
axs[1].set_xlabel('Iteration')
plt.tight_layout()
plt.show()

## Compare against vanilla HMC

Both samplers see the same number of post-warmup samples. NUTS automatically picks longer trajectories where they pay off, giving higher ESS per iteration than fixed-length HMC.

In [ ]:
hmc = HamiltonianMonteCarlo(target=target, x0=x0, step_size=0.2, num_steps=5, generator=torch.Generator().manual_seed(0))
hmc_samples, _ = hmc(N, burn_in=warmup)

print(f'NUTS acceptance (avg tree): {nuts.last_alpha:.3f}; final step size {nuts.dynamics.step_size:.3f}')
print(f'HMC  acceptance: {100*hmc.acceptance_ratio:.1f}%')

print('NUTS ESS:', diagnostics.effective_sample_size(samples).tolist())
print('HMC  ESS:', diagnostics.effective_sample_size(hmc_samples).tolist())

In [ ]:
for s, label in [(samples, 'NUTS'), (hmc_samples, 'HMC')]:
    vis = SampleVisualizer(s)
    vis.triangular_hist(bins=50, labels=['$x_1$', '$x_2$'], titles=[label])
plt.show()